# Phase 4A — Continue Full-Dataset Pretraining

Resume the 800-image MSE run from its **epoch-20** checkpoint, including Adam state. This experiment adds 20 epochs with the original hyperparameters and writes to a separate Drive directory.

Compare the original best generator, the best generator after continuation, and bicubic on the **same 50 validation crops**. Use a GPU runtime. The first new epoch reports its duration.


## 1. Update the project
Use the current main branch and the dependencies from the earlier notebooks.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path("/content/SRGAN-Image-Super-Resolution")
REPO_URL = "https://github.com/ennaguiyoussef/SRGAN-Image-Super-Resolution.git"
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    if not (PROJECT_ROOT / ".git").is_dir():
        raise RuntimeError("Project folder exists but is not a Git checkout.")
    changes = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain", "--untracked-files=no"],
        text=True,
    )
    if changes.strip():
        raise RuntimeError("Save local tracked edits before updating this checkout.")
    branch = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "branch", "--show-current"], text=True
    ).strip()
    if branch != "main":
        raise RuntimeError("Select the main branch before running this notebook.")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Project checkout ready.")


## 2. Restore the 800 training images and 50 validation images
Use the same image IDs, crops, and Drive cache as the original run.


In [ ]:
import yaml
import torch
from src.setup import environment_report, load_config, seed_everything

config = load_config(PROJECT_ROOT / "configs/srgan.yaml")
pretrain = yaml.safe_load((PROJECT_ROOT / "configs/pretrain_full.yaml").read_text())
if pretrain["smoke_mode"] is not False:
    raise ValueError("This notebook requires the full 800-image dataset.")
if pretrain["num_workers"] != 0:
    raise ValueError("This phase uses num_workers=0 for epoch-boundary replay.")
report = environment_report()
print(json.dumps(report, indent=2))
if not report["cuda_available"]:
    raise RuntimeError("Select a GPU runtime before training.")
device = torch.device("cuda:0")
seed_everything(config["seed"])
print(json.dumps(pretrain, indent=2))
from google.colab import drive
from src.setup import prepare_directories
from src.data import extract_hr_archive, split_manifest
from src.dataset import SuperResolutionDataset

drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/SRGAN-Image-Super-Resolution")
paths = prepare_directories(PROJECT_ROOT, OUTPUT_ROOT)
DATA_ROOT = paths["data"] / "DIV2K"
manifest_path = OUTPUT_ROOT / "splits" / "div2k_seed42.json"
if not manifest_path.is_file():
    raise RuntimeError("Complete phase 2 first: split manifest is missing.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest != split_manifest(config["seed"]):
    raise RuntimeError("Split manifest does not match the current seed.")
train_names = manifest["train"]
assert len(train_names) == 800
validation_names = manifest["validation"]
for folder, names in [("DIV2K_train_HR", train_names), ("DIV2K_valid_HR", validation_names)]:
    missing = [name for name in names if not (DATA_ROOT / name).is_file()]
    if missing:
        archive = OUTPUT_ROOT / "dataset_cache" / f"{folder}.zip"
        if not archive.is_file():
            raise RuntimeError(f"Missing archive {archive.name}. Complete phase 2 first.")
        extract_hr_archive(archive, DATA_ROOT, folder, [int(Path(name).stem) for name in missing])

train_dataset = SuperResolutionDataset(
    DATA_ROOT, train_names, hr_size=config["hr_crop_size"], scale=config["scale_factor"], training=True,
)
validation_dataset = SuperResolutionDataset(
    DATA_ROOT, validation_names, hr_size=config["hr_crop_size"], scale=config["scale_factor"], training=False,
)
print("Training images:", len(train_dataset))
print("Validation images:", len(validation_dataset))
print("Held-out test images are not loaded.")


## 3. Create or resume the continuation checkpoint
The first execution copies the epoch-20 model, Adam state, and history into a new run. It also stores the original best weights for comparison.

Subsequent executions resume the continuation run from its latest complete epoch. Increase `TARGET_EPOCHS` to extend it.


In [ ]:
import hashlib
from src.pretrain import atomic_torch_save

TARGET_EPOCHS = 40
SOURCE_EPOCH = 20
source_run = f"{pretrain['run_name']}_full800"
source_path = paths["checkpoints"] / source_run / "last.pt"
RUN_ID = "pretrain_full_continue_v1"
RUN_DIR = paths["checkpoints"] / RUN_ID
RESULTS_DIR = paths["results"] / RUN_ID
destination = RUN_DIR / "last.pt"
continuing = destination.is_file()

if continuing:
    saved = torch.load(destination, map_location="cpu", weights_only=True)
    continuation = saved.get("continuation")
    if not continuation or continuation["source_run"] != source_run:
        raise ValueError("The destination belongs to a different continuation run.")
else:
    if not source_path.is_file():
        raise FileNotFoundError(f"Restore the full-data epoch-20 checkpoint at {source_path}.")
    saved = torch.load(source_path, map_location="cpu", weights_only=True)
    if saved.get("format_version") != 1 or saved["epoch"] != SOURCE_EPOCH:
        raise ValueError(f"Expected the original full-data checkpoint at epoch {SOURCE_EPOCH}.")
    continuation = {
        "source_run": source_run, "source_epoch": SOURCE_EPOCH,
        "source_best_epoch": saved["best_epoch"], "source_best_model": saved["best_model"],
    }
    saved["continuation"] = continuation

settings = saved["settings"]
experiment = settings["experiment"]
if (
    settings["train_images"] != len(train_dataset)
    or settings["validation_images"] != len(validation_dataset)
    or experiment.get("train_names") != train_names
    or experiment.get("validation_names") != validation_names
    or experiment.get("scale_factor") != config["scale_factor"]
    or experiment.get("hr_crop_size") != config["hr_crop_size"]
    or experiment.get("manifest_sha256") != hashlib.sha256(manifest_path.read_bytes()).hexdigest()
):
    raise ValueError("The saved training and validation protocol differs from the current data.")

if not continuing:
    atomic_torch_save(saved, destination)
    print("Copied the epoch-20 model and Adam state into the continuation run.")
print("Saved epoch:", saved["epoch"])
print("Original best epoch:", continuation["source_best_epoch"])
print("Continuation directory:", RUN_DIR)
del saved


## 4. Continue training to epoch 40
The continuation uses the original learning rate and batch size. Selection uses validation MSE and considers every epoch from the original run and its continuation.


In [ ]:
from src.models import Generator
from src.pretrain import fit_pretrain

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
generator = Generator()
state = fit_pretrain(
    generator, train_dataset, validation_dataset, RUN_DIR,
    epochs=TARGET_EPOCHS, batch_size=settings["batch_size"],
    learning_rate=settings["learning_rate"], seed=settings["seed"],
    gradient_clip_norm=settings["gradient_clip_norm"], device=device,
    experiment=experiment, source_commit=commit,
    initial_checkpoint=settings.get("initial_checkpoint"),
)
print("Completed epoch:", state["epoch"])
print("Best validation epoch:", state["best_epoch"])


## 5. Compare with the original generator and bicubic
Compute RGB PSNR and SSIM on the same fixed 96×96 validation center crops, including all crop pixels. The phase-5 full-image test uses a separate protocol.

The table compares the original best weights with the best weights after continuation. The plot shows PSNR from epoch 20 onward with the bicubic reference.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader
from src.evaluation import image_metrics, rgb_array
from src.pretrain import atomic_json_save

loader = DataLoader(validation_dataset, batch_size=settings["batch_size"], num_workers=0)

def bicubic_image(lr, hr):
    pixels = np.rint(lr.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return rgb_array(Image.fromarray(pixels).resize((hr.shape[2], hr.shape[1]), Image.Resampling.BICUBIC))

@torch.inference_mode()
def score_validation(weights=None):
    if weights is not None:
        generator.load_state_dict(weights)
        generator.eval()
    scores = []
    for lr, hr in loader:
        reference = ((hr + 1) / 2).permute(0, 2, 3, 1).numpy()
        if weights is None:
            predictions = [bicubic_image(lr_img, hr_img) for lr_img, hr_img in zip(lr, hr)]
        else:
            sr = generator(lr.to(device)).cpu()
            predictions = ((sr + 1) / 2).clamp(0, 1).permute(0, 2, 3, 1).numpy()
        scores.extend(image_metrics(target, prediction, border=0)
                      for target, prediction in zip(reference, predictions))
    return {key: float(np.mean([row[key] for row in scores])) for key in ("psnr_db", "ssim")}

baseline = score_validation()
original = score_validation(continuation["source_best_model"])
best = score_validation(state["best_model"])
summary = {"bicubic": baseline, "original_best": original, "continuation_best": best}
print(f"{'Method':<24} {'PSNR (dB)':>10} {'SSIM':>10}")
for name, metrics in summary.items():
    print(f"{name:<24} {metrics['psnr_db']:>10.3f} {metrics['ssim']:>10.5f}")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
history = [row for row in state["history"] if row["epoch"] >= SOURCE_EPOCH]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([r["epoch"] for r in history], [r["val_psnr_rgb"] for r in history], marker="o", label="Generator")
ax.axhline(baseline["psnr_db"], color="#555555", linestyle="--", label="Bicubic")
ax.axhline(original["psnr_db"], color="#287258", linestyle=":", label="Original best")
ax.set(xlabel="Full-data epoch", ylabel="Mean RGB PSNR (dB)", title="Validation center crops")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "continuation_psnr.png", dpi=160)
plt.show()

atomic_json_save({
    "git_commit": commit, "source_run": source_run,
    "source_epoch": SOURCE_EPOCH, "source_best_epoch": continuation["source_best_epoch"],
    "completed_epoch": state["epoch"], "best_epoch": state["best_epoch"],
    "validation_images": len(validation_dataset), "border": 0, "summary": summary,
}, RESULTS_DIR / "continuation_report.json")
print("Continuation complete. Saved metrics in", RESULTS_DIR)
